In [1]:
# TrendPulse - Part 1: Python Foundations
## 1. Setup

In [2]:
from datetime import datetime

# ---------- Configuration ----------
DATA_FOLDER = "data/"                      # str
COUNTRIES = ["US", "GB", "IN"]             # list of str
CATEGORIES_FILE = DATA_FOLDER + "categories.csv"
MAX_ROWS = None                            # None = load all rows
VIRAL_VIEWS = 1_000_000                    # int (threshold, you can change it later)
MIN_LIKE_RATIO = 0.80                      # float (threshold, you can change it later)

# ---------- Run header ----------
start_time = datetime.now()
print("=" * 40)
print(f"Countries to load: {', '.join(COUNTRIES)}")
print(f"Run started: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 40)

Countries to load: US, GB, IN
Run started: 2026-11-21 18:03:08


### Notebook discipline
- Each section has a clear heading.
- All settings are kept in variables at the top.
- I used Runtime -> Restart and run all to check that the notebook runs from top to bottom.
- No hidden state: cells do not depend on variables created in deleted or out-of-order cells.

## 2. Loading data

In [3]:
import pandas as pd

REQUIRED_COLUMNS = [
    "video_id", "trending_date", "title", "channel_title", "category_id",
    "publish_time", "views", "likes", "dislikes", "comment_count",
    "comments_disabled", "ratings_disabled"
]

def load_country(path, country, max_rows=None):
    df = pd.read_csv(path, nrows=max_rows)

    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        print(f"{country}: missing columns {missing}")
        return []

    df["country"] = country
    return df.to_dict("records")

In [4]:
all_records = []
records_by_country = {}

for country in COUNTRIES:
    path = f"{DATA_FOLDER}{country}videos.csv"
    rows = load_country(path, country, MAX_ROWS)
    records_by_country[country] = rows
    all_records.extend(rows)
    print(f"{country}: {len(rows)} records loaded")

print(f"Total records: {len(all_records)}")

US: 16080 records loaded
GB: 15075 records loaded
IN: 15075 records loaded
Total records: 46230


In [5]:
cat_df = pd.read_csv(CATEGORIES_FILE)
print(cat_df.columns.tolist())

['category_id', 'category_name']


In [6]:
category_names = {}
for row in cat_df.to_dict("records"):
    category_names[row["category_id"]] = row["category_name"]

print(list(category_names.items())[:5])

[(1, 'Film & Animation'), (2, 'Autos & Vehicles'), (10, 'Music'), (15, 'Pets & Animals'), (17, 'Sports')]


In [7]:
print(all_records[0])

{'video_id': 'g_N-w8LeAi8', 'trending_date': '17.14.11', 'title': 'Honest Football Prank LEAKED', 'channel_title': 'QuietSpeech10', 'category_id': 22, 'publish_time': '2017-11-14T07:04:28.000Z', 'tags': 'Monsoon|Robots', 'views': 48555, 'likes': 2189.0, 'dislikes': 180, 'comment_count': 136.0, 'thumbnail_link': 'https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg', 'comments_disabled': False, 'ratings_disabled': False, 'video_error_or_removed': False, 'description': 'Watch this video. Subscribe for more content about chess. Business enquiries in the channel about page.', 'country': 'US'}


## 3. Cleaning

In [8]:
from datetime import datetime

failures = {
    "views": 0, "likes": 0, "dislikes": 0, "comment_count": 0,
    "comments_disabled": 0, "ratings_disabled": 0,
    "trending_date": 0, "publish_time": 0
}

def to_int(value, field):
    try:
        return int(value)
    except (ValueError, TypeError):
        failures[field] += 1
        return 0

def to_bool(value, field):
    text = str(value).strip().lower()
    if text == "true":
        return True
    elif text == "false":
        return False
    else:
        failures[field] += 1
        return False

def to_date(text, date_format, field):
    try:
        return datetime.strptime(str(text), date_format).date()
    except ValueError:
        failures[field] += 1
        return None

In [9]:
clean_records = []
dropped = 0

for row in all_records:
    trending = to_date(row["trending_date"], "%y.%d.%m", "trending_date")
    published = to_date(str(row["publish_time"])[:10], "%Y-%m-%d", "publish_time")

    if trending is None or published is None:
        dropped += 1
        continue

    row["trending_date"] = trending
    row["publish_time"] = published
    row["views"] = to_int(row["views"], "views")
    row["likes"] = to_int(row["likes"], "likes")
    row["dislikes"] = to_int(row["dislikes"], "dislikes")
    row["comment_count"] = to_int(row["comment_count"], "comment_count")
    row["comments_disabled"] = to_bool(row["comments_disabled"], "comments_disabled")
    row["ratings_disabled"] = to_bool(row["ratings_disabled"], "ratings_disabled")

    clean_records.append(row)

print(f"Records before cleaning: {len(all_records)}")
print(f"Records after cleaning:  {len(clean_records)}")
print(f"Dropped (bad dates):     {dropped}")
print("Parse failures per field:")
for field, count in failures.items():
    print(f"  {field}: {count}")

Records before cleaning: 46230
Records after cleaning:  46230
Dropped (bad dates):     0
Parse failures per field:
  views: 0
  likes: 12
  dislikes: 0
  comment_count: 36
  comments_disabled: 0
  ratings_disabled: 0
  trending_date: 0
  publish_time: 0


In [10]:
print(len(clean_records) >= 40000)

True


## 4. Data structures

In [11]:
# clean_records is already a list of dictionaries
print(type(clean_records), type(clean_records[0]))

# Tuples for a few key fields
key_tuples = []
for row in clean_records:
    item = (row["video_id"], row["trending_date"], row["views"], row["likes"])
    key_tuples.append(item)

print(key_tuples[0])

<class 'list'> <class 'dict'>
('g_N-w8LeAi8', datetime.date(2017, 11, 14), 48555, 2189)


In [12]:
ids_by_country = {}
for country in COUNTRIES:
    ids_by_country[country] = set()

for row in clean_records:
    ids_by_country[row["country"]].add(row["video_id"])

for country in COUNTRIES:
    print(f"{country}: {len(ids_by_country[country])} unique videos")

# Intersection: in more than one country
us_and_gb = ids_by_country["US"] & ids_by_country["GB"]
in_all = ids_by_country["US"] & ids_by_country["GB"] & ids_by_country["IN"]

# Difference: only in one country
only_us = ids_by_country["US"] - ids_by_country["GB"] - ids_by_country["IN"]

print(f"In both US and GB: {len(us_and_gb)}")
print(f"In all three: {len(in_all)}")
print(f"Only in US: {len(only_us)}")

US: 5068 unique videos
GB: 4113 unique videos
IN: 5878 unique videos
In both US and GB: 134
In all three: 20
Only in US: 4834


In [13]:
channel_counts = {}
for row in clean_records:
    channel = row["channel_title"]
    if channel in channel_counts:
        channel_counts[channel] += 1
    else:
        channel_counts[channel] = 1

def get_count(pair):
    return pair[1]

sorted_channels = sorted(channel_counts.items(), key=get_count, reverse=True)
top_15 = sorted_channels[:15]

for name, count in top_15:
    print(f"{name}: {count} appearances")

QuietChallenge75: 2464 appearances
OfficialReview41: 1586 appearances
UltimateUnboxing65: 1433 appearances
InsaneReaction46: 1233 appearances
Brand NewChallenge45: 1175 appearances
CrazyTrailer91: 1018 appearances
FinalTrailer17: 911 appearances
UltimateSpeech20: 770 appearances
Brand NewSecret51: 684 appearances
WildHighlights7: 682 appearances
SurprisingHighlights41: 650 appearances
FinalBreakdown11: 643 appearances
CrazyRecipe67: 603 appearances
QuietInterview9: 570 appearances
UltimateVlog96: 565 appearances


In [14]:
summary = {}
for row in clean_records:
    country = row["country"]
    category = category_names.get(row["category_id"], "Unknown")

    if country not in summary:
        summary[country] = {}
    if category not in summary[country]:
        summary[country][category] = 0
    summary[country][category] += 1

for country in COUNTRIES:
    print(f"--- {country} ---")
    for category, count in list(summary[country].items())[:5]:
        print(f"  {category}: {count}")

--- US ---
  People & Blogs: 1423
  News & Politics: 755
  Music: 3052
  Education: 561
  Entertainment: 4371
--- GB ---
  Science & Technology: 470
  Howto & Style: 588
  Comedy: 941
  Entertainment: 2844
  Music: 5013
--- IN ---
  Music: 2298
  Film & Animation: 770
  Entertainment: 5117
  Comedy: 966
  People & Blogs: 1098


## 5. Variable scope

In [15]:
outside_var = "I am defined outside the function"

def scope_demo():
    inside_var = "I am defined inside the function"
    print("Inside the function, I can read:", outside_var)
    print("Inside the function, I can read:", inside_var)

scope_demo()

try:
    print(inside_var)
except NameError as error:
    print("Outside the function, this fails:", error)

Inside the function, I can read: I am defined outside the function
Inside the function, I can read: I am defined inside the function
Outside the function, this fails: name 'inside_var' is not defined


## 6. Loops

In [16]:
def title_features(title):
    title = str(title)
    words = title.split()

    uppercase_words = 0
    for word in words:
        if word.isupper():
            uppercase_words += 1

    punctuation_marks = 0
    for character in title:
        if character == "!" or character == "?":
            punctuation_marks += 1

    return {
        "uppercase_words": uppercase_words,
        "punctuation_marks": punctuation_marks,
        "word_count": len(words)
    }

print(title_features("WE WANT TO TALK ABOUT OUR MARRIAGE!"))
print(title_features("Why is this so good?!"))

{'uppercase_words': 7, 'punctuation_marks': 1, 'word_count': 7}
{'uppercase_words': 0, 'punctuation_marks': 2, 'word_count': 5}


In [17]:
for i in range(5):
    title = clean_records[i]["title"]
    print(i, title_features(title))

0 {'uppercase_words': 1, 'punctuation_marks': 0, 'word_count': 4}
1 {'uppercase_words': 0, 'punctuation_marks': 0, 'word_count': 3}
2 {'uppercase_words': 1, 'punctuation_marks': 0, 'word_count': 6}
3 {'uppercase_words': 0, 'punctuation_marks': 2, 'word_count': 6}
4 {'uppercase_words': 0, 'punctuation_marks': 0, 'word_count': 3}


In [18]:
index = 0
first_big_video = None

while index < len(clean_records):
    row = clean_records[index]

    if row["views"] == 0:
        index += 1
        continue

    if row["views"] > VIRAL_VIEWS:
        first_big_video = row
        break

    index += 1

if first_big_video is not None:
    print(f"First video above {VIRAL_VIEWS:,} views is at position {index}:")
    print(first_big_video["title"], "-", first_big_video["views"])
else:
    print("No video found above the threshold.")

First video above 1,000,000 views is at position 7:
Official Cats Speech EXCLUSIVE - 2841278


## 7. Classification

In [20]:
VIRAL_VIEWS = 1_000_000     # already set in Setup; kept here for clarity
STEADY_VIEWS = 100_000
MIN_LIKE_RATIO = 0.80

In [19]:
def classify_video(record):
    views = record["views"]
    likes = record["likes"]
    dislikes = record["dislikes"]
    total_ratings = likes + dislikes

    # Edge cases first
    if record["ratings_disabled"] or total_ratings == 0:
        return "Unrated"
    if views == 0:
        return "Low Engagement"

    like_ratio = likes / total_ratings   # safe: total_ratings is not 0 here

    if record["comments_disabled"] and views < STEADY_VIEWS:
        return "Low Engagement"

    if views >= VIRAL_VIEWS:
        if like_ratio >= MIN_LIKE_RATIO:
            return "Viral Hit"
        else:
            return "Divisive"
    elif views >= STEADY_VIEWS:
        if like_ratio >= MIN_LIKE_RATIO:
            return "Steady Performer"
        else:
            return "Divisive"
    else:
        return "Low Engagement"

In [21]:
label_counts = {}

for row in clean_records:
    label = classify_video(row)
    row["label"] = label
    if label in label_counts:
        label_counts[label] += 1
    else:
        label_counts[label] = 1

for label, count in label_counts.items():
    print(f"{label}: {count}")
print("Total:", sum(label_counts.values()))

Low Engagement: 8738
Divisive: 1268
Steady Performer: 26183
Viral Hit: 9497
Unrated: 544
Total: 46230


In [22]:
example = clean_records[7]
print(example["title"])
print(example["views"], example["likes"], example["dislikes"])
print(example["ratings_disabled"], example["comments_disabled"])
print(example["label"])

Official Cats Speech EXCLUSIVE
2841278 52635 7691
False False
Viral Hit


In [ ]:
if label in label_counts:
    label_counts[label] += 1
else:
    label_counts[label] = 1

## 8. Refactor

In [24]:
def add_count(counter, key):
    if key in counter:
        counter[key] += 1
    else:
        counter[key] = 1

In [25]:
channel_counts = {}
for row in clean_records:
    add_count(channel_counts, row["channel_title"])

print(len(channel_counts), "channels counted")

438 channels counted


In [26]:
label_counts = {}
for row in clean_records:
    add_count(label_counts, row["label"])

for label, count in label_counts.items():
    print(f"{label}: {count}")
print("Total:", sum(label_counts.values()))

Low Engagement: 8738
Divisive: 1268
Steady Performer: 26183
Viral Hit: 9497
Unrated: 544
Total: 46230


In [27]:
import os
os.makedirs("data_small", exist_ok=True)

for country in COUNTRIES:
    df = pd.read_csv(f"data/{country}videos.csv")
    df = df.drop(columns=["description", "tags", "thumbnail_link"], errors="ignore")
    df.to_csv(f"data_small/{country}videos.csv", index=False)

print(os.listdir("data_small"))

['GBvideos.csv', 'USvideos.csv', 'INvideos.csv']


In [28]:
import shutil
shutil.make_archive("data_small", "zip", "data_small")

'/content/data_small.zip'